# Scheduled quantum HUGR generation

Alpha models qubits as first-class linear values over polyhedral domains. This notebook builds a scheduled quantum program, specializes its parameters, and emits a validated HUGR with concrete array boundaries.

In [1]:
import alphalang

In [2]:
%%alphalang system

affine QuantumPipeline [T,N] -> {:T>0 and N>0}
inputs
    linear A0, B0 : {[i] : 0 <= i < N} of qubit;
outputs
    M : {[i] : 0 <= i < N} of bool;
locals
    linear Q : {[t,i] : 0 <= t < T and 0 <= i < N} of qubit;
    linear A1, B1 : {[i] : 0 <= i < N} of qubit;
let
    over {[t,i] : t=0 and 0<=i<N} with [t,i] : (Q[t,i]) = qalloc();
    over {[t,i] : 0<t<T and 0<=i<N} with [t,i] : (Q[t,i]) = h(Q[t-1,i]);
    with [i] : (M[i]) = measure(Q[T-1,i]);
    with [i] : (A1[i], B1[i]) = cx(A0[i], B0[i]);
    with [i] : () = discard(A1[i]);
    with [i] : () = discard(B1[i]);
.

In [3]:
normalized = alphalang.normalize(system)

[(variable.name, variable.element_type, variable.multiplicity) for variable in normalized.inputs + normalized.outputs + normalized.locals]

[('A0', ElementType.QUBIT, Multiplicity.LINEAR),
 ('B0', ElementType.QUBIT, Multiplicity.LINEAR),
 ('M', ElementType.BOOL, Multiplicity.UNRESTRICTED),
 ('Q', ElementType.QUBIT, Multiplicity.LINEAR),
 ('A1', ElementType.QUBIT, Multiplicity.LINEAR),
 ('B1', ElementType.QUBIT, Multiplicity.LINEAR)]

## Schedule and specialize

A legal schedule keeps each qubit producer before its consumer. The alternate mapping changes the independent CX chain's placement without changing resource flow.

In [4]:
SCHEDULE = """[T,N] -> {
Q__call0[t,i] -> [t,0,i]; Q__call1[t,i] -> [t,1,i]; M__call0[i] -> [T,2,i];
A1__call0[i] -> [T,3,i]; discard__call0[i] -> [T,4,i]; discard__call1[i] -> [T,5,i]
}"""
ALTERNATE_SCHEDULE = """[T,N] -> {
A1__call0[i] -> [0,0,i]; discard__call0[i] -> [0,1,i]; discard__call1[i] -> [0,2,i];
Q__call0[t,i] -> [t+1,0,i]; Q__call1[t,i] -> [t+1,1,i]; M__call0[i] -> [T+1,2,i]
}"""

scheduled = normalized.schedule(SCHEDULE)
alternate = normalized.schedule(ALTERNATE_SCHEDULE)
print(repr(scheduled))
print("alternate schedule valid:", isinstance(alternate, alphalang.ScheduledSystem))

[T, N] -> { M__call0[i] -> [T, 2, i] : 0 <= i < N; A1__call0[i] -> [T, 3, i] : 0 <= i < N; discard__call0[i] -> [T, 4, i] : 0 <= i < N; discard__call1[i] -> [T, 5, i] : 0 <= i < N; Q__call0[t = 0, i] -> [0, 0, i] : 0 <= i < N; Q__call1[t, i] -> [t, 1, i] : 0 < t < T and 0 <= i < N }
alternate schedule valid: True


In [5]:
bindings = {"T": 3, "N": 4}
envelope = alphalang.generate_hugr(scheduled, bindings)
alternate_envelope = alphalang.generate_hugr(alternate, bindings)
print("prefix:", envelope[:8])
print("characters:", len(envelope))
print("serialized HUGR:", "HUGRiHJ" in envelope)
print("alternate differs:", alternate_envelope != envelope)

prefix: HUGRiHJv
characters: 134739
serialized HUGR: True
alternate differs: True


## Wrap the Alpha HUGR with Guppy

The Guppy entry point allocates two four-qubit registers and calls a declaration whose `@link_name` is the symbol replaced by Alpha. `@ owned` makes the declaration consume those linear arrays, matching `QuantumPipeline`'s input contract.

In [6]:
from pathlib import Path

from guppylang import guppy
from guppylang.library import link_name
from guppylang.std.builtins import array
from guppylang.std.quantum import owned, qubit
from hugr.package import Package


@guppy.declare
@link_name("quantum_pipeline")
def quantum_pipeline(
    a: array[qubit, 4] @ owned, b: array[qubit, 4] @ owned
) -> array[bool, 4]: ...


@guppy
@link_name("main")
def guppy_main() -> array[bool, 4]:
    a = array(qubit() for _ in range(4))
    b = array(qubit() for _ in range(4))
    return quantum_pipeline(a, b)


# Compile Alpha and Guppy independently, then replace the declaration.
alpha_quantum_hugr = alphalang.generate_hugr(scheduled, {"T": 3, "N": 4})
guppy_wrapper = guppy_main.compile()
wrapped_quantum = alphalang.link_alpha_function(
    guppy_wrapper, alpha_quantum_hugr, symbol="quantum_pipeline"
)

repo_root = next(
    root
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "alphalang" / "notebooks").is_dir()
)
artifact_dir = repo_root / "alphalang" / "notebooks" / "artifacts"
artifact_dir.mkdir(exist_ok=True)
quantum_wrapper_path = artifact_dir / "quantum_pipeline_wrapped.hugr"
quantum_wrapper_path.write_bytes(wrapped_quantum.to_bytes())
Package.from_bytes(quantum_wrapper_path.read_bytes())
print("wrapped HUGR:", quantum_wrapper_path.resolve())

wrapped HUGR: /Users/louis.narmour/git/poly/alpha/alphalang/notebooks/artifacts/quantum_pipeline_wrapped.hugr


### Replace a retained dummy

A concrete dummy can be useful when the wrapper must compile before the Alpha implementation is available. Guppy's direct entry-point compilation may inline that dummy, so this example compiles both functions as library members, selects `prepared_bits_main` as the entry point, and then replaces `prepared_bits`.

In [7]:
%%alphalang prepared_system

affine PreparedBits [N] -> {:N>0}   
    outputs 
        M : {[i] : 0 <= i < N} of bool;
    locals 
        linear Q : {[i] : 0 <= i < N} of qubit;
let
    with [i] : (Q[i]) = qalloc();
    with [i] : (M[i]) = measure(Q[i]);
.

In [8]:
from guppylang.library import GuppyLibrary
from hugr.ops import FuncDefn

PREPARED_BITS_SCHEDULE = """[N] -> {
Q__call0[i] -> [0,i]; M__call0[i] -> [1,i]
}"""

prepared_normalized = alphalang.normalize(prepared_system)
prepared_schedule = prepared_normalized.schedule(PREPARED_BITS_SCHEDULE)
prepared_alpha_hugr = alphalang.generate_hugr(prepared_schedule, {"N": 4})

@guppy
@link_name("prepared_bits")
def prepared_bits() -> array[bool, 4]:
    return array(False, False, False, False)

@guppy
@link_name("prepared_bits_main")
def prepared_bits_main() -> array[bool, 4]:
    return prepared_bits()

dummy_wrapper = GuppyLibrary.from_members(
    prepared_bits, prepared_bits_main
).compile()
dummy_module = dummy_wrapper.modules[0]
dummy_module.entrypoint = next(
    node
    for node in dummy_module.children(dummy_module.module_root)
    if isinstance(dummy_module[node].op, FuncDefn)
    and dummy_module[node].op.f_name == "prepared_bits_main"
)

wrapped_dummy = alphalang.link_alpha_function(
    dummy_wrapper, prepared_alpha_hugr, symbol="prepared_bits"
)

dummy_wrapper_path = artifact_dir / "prepared_bits_wrapped.hugr"
dummy_wrapper_path.write_bytes(wrapped_dummy.to_bytes())
Package.from_bytes(dummy_wrapper_path.read_bytes())
print("wrapped HUGR:", dummy_wrapper_path.resolve())

wrapped HUGR: /Users/louis.narmour/git/poly/alpha/alphalang/notebooks/artifacts/prepared_bits_wrapped.hugr


## Rejected programs

Qubits must be linear, and the compact HUGR realization requires zero-based rectangular resource roots.

In [9]:
NON_LINEAR_QUBIT = """affine Invalid [N] -> {:N>0}
inputs Q : {[i] : 0 <= i < N} of qubit;
outputs M : {[i] : 0 <= i < N} of bool;
let with [i] : (M[i]) = measure(Q[i]);
.
"""

try:
    alphalang.parse(NON_LINEAR_QUBIT)
except ValueError as error:
    print(str(error).splitlines()[-1])

qubit variable 'Q' must be declared linear


In [10]:
TRIANGULAR = """affine Triangular [N] -> {:N>0}
inputs linear Q0 : {[i,j] : 0 <= i < N and 0 <= j <= i} of qubit;
outputs linear Q1 : {[i,j] : 0 <= i < N and 0 <= j <= i} of qubit;
let with [i,j] : (Q1[i,j]) = h(Q0[i,j]);
.
"""

triangular = alphalang.normalize(alphalang.parse(TRIANGULAR))
try:
    alphalang.generate_hugr(triangular, {"N": 3})
except ValueError as error:
    print(error)

realization failed: root domain is not rectangular: { [i, j] : 0 <= i <= 2 and 0 <= j <= i }


## Current boundary

The backend emits concrete HUGRs after all parameters are bound. Measurement-dependent control and parametric HUGR signatures are deferred.